# 01 — データ探索・EDA・検証

- 保存済みデータの品質・分布・分割を確認する。
- 処理は `src/`、Notebookは分析・可視化を担当。
- 探索・性能比較は02_analysis、選択した推薦の確認は03_recommendation。

## 1. Overview

| 対象 | 内容・規模 |
| --- | --- |
| PostgreSQL | `MUSIC_RECSYS_SNAPSHOT_ID` で公開済みsnapshotを指定。件数は読込み後に表示 |
| 保存済み分割 | `MUSIC_RECSYS_SPLIT_ID` で指定。snapshot未指定時は分割に結び付くものを使用 |
| 既定モード | 合成fixture：4曲・2ユーザー。実Music4Allではない |
| 参考資料 | 全量イベントの保存済みEDA。選択したsnapshot・分割の集計とは別に表示 |
| 出典 | [Zenodo record 15394646](https://zenodo.org/records/15394646) |

- 起動：`uv run --locked --group data --group notebook --group tracking jupyter lab`
- DBを読む場合は起動時に `.env` を渡す。接続値はセルに書かない。
- 既定実行はDB・配布ファイル・常駐MLflow不要。


In [ ]:
import json
import os
from datetime import datetime
from pathlib import Path
from uuid import UUID

import matplotlib.pyplot as plt
from IPython.display import display
from pandas import DataFrame

from music_recsys.analysis import split_candidate_tables, temporal_rows
from music_recsys.experiment_search import load_study_config
from music_recsys.search_analysis import validation_overview
from music_recsys.snapshot_analysis import (
    load_fixture,
    load_snapshot_frames,
    snapshot_eda,
    validation_rows,
)

ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "configs/toy.toml").is_file()
)
SNAPSHOT_ID = os.environ.get("MUSIC_RECSYS_SNAPSHOT_ID")
SPLIT_ID = os.environ.get("MUSIC_RECSYS_SPLIT_ID")
STORAGE_ROOT = ROOT / "data/imports"
DATASET_KEY = "music4all-onion"
VERSION_LABEL = "v2"
FIXTURE_PATH = ROOT / "examples/catalog/snapshot.json"
PROFILE_PATH = Path(
    os.environ.get("MUSIC_RECSYS_PROFILE", str(ROOT / "experiments/music4all/eda/profile.json"))
)

mode = "PostgreSQL saved data" if SNAPSHOT_ID or SPLIT_ID else "Saved synthetic fixture"
display({"mode": mode, "snapshot_id": SNAPSHOT_ID, "split_id": SPLIT_ID})
EXPERIMENT_CONFIG = Path(
    os.environ.get("MUSIC_RECSYS_EXPERIMENT_CONFIG", ROOT / "configs/workflow-toy.toml")
)
GROUND_TRUTH_PROFILE = Path(
    os.environ.get(
        "MUSIC_RECSYS_GROUND_TRUTH_PROFILE",
        ROOT / "experiments/music4all/eda/ground-truth-profile.json",
    )
)
WORKFLOW = load_study_config(EXPERIMENT_CONFIG)
print("01→02→03の共有設定:", EXPERIMENT_CONFIG)

## 2. Data Loading

| モード | 読込み |
| --- | --- |
| PostgreSQL | 指定した公開済みsnapshotのSQL結果をDataFrameへ |
| fixture | 保存済み合成データをDataFrameへ |
| 全量イベントの参考資料 | 保存済み集計JSONのみ |

- SQLは読み取り専用。版・snapshotを固定し、Parquetも照合する。
- 表示は規模・カラム・出典。取り込み・分割処理は実行しない。

In [ ]:
saved_split = None
if SNAPSHOT_ID or SPLIT_ID:
    from music_recsys.ingestion.postgres import engine_from_environment

    engine = engine_from_environment()
    try:
        if SPLIT_ID:
            from music_recsys.ingestion.splits import load_split

            saved_split = load_split(engine, STORAGE_ROOT, UUID(SPLIT_ID))
            if SNAPSHOT_ID and UUID(SNAPSHOT_ID) != saved_split["snapshot_id"]:
                raise ValueError("selected split belongs to a different snapshot")
            SNAPSHOT_ID = str(saved_split["snapshot_id"])
        snapshot = load_snapshot_frames(
            engine,
            STORAGE_ROOT,
            UUID(SNAPSHOT_ID),
            dataset_key=DATASET_KEY,
            version_label=VERSION_LABEL,
        )
    finally:
        engine.dispose()
else:
    snapshot = load_fixture(FIXTURE_PATH)

tracks_df, users_df = snapshot.tracks, snapshot.users
profile = json.loads(PROFILE_PATH.read_text(encoding="utf-8"))
display(snapshot.overview)
display(
    {
        "tracks_frame": {"rows": len(tracks_df), "columns": list(tracks_df.columns)},
        "users_frame": {"rows": len(users_df), "columns": list(users_df.columns)},
        "reference_eda": {
            key: profile[key] for key in ("n_events", "n_users", "n_items", "min_time", "max_time")
        },
    }
)

## 3. Data Validation

| 確認 | 内容 |
| --- | --- |
| 件数・ID | 元ID／UUIDの欠損・重複・形式 |
| タグ | `payload=NULL` は未取得、`tags={}` は取得済み・タグなし |
| 実DB | 所属・参照・作成run・公開状態・manifest／fingerprint |
| Parquet | hash・schema・全行内容 |

- DB検証失敗は読込み時にエラー。
- fixtureのDB／Parquet検証は `not checked`。
- `expected=None` の欠損タグ数・空タグ数は観測値。存在だけではエラーにしない。

In [ ]:
validation = validation_rows(
    snapshot,
    expected_tracks=snapshot.overview.get("n_tracks", 4),
    expected_users=snapshot.overview.get("n_users", 2),
)
display(DataFrame(validation))

## 4. EDA

| 図 | 集計対象・定義 |
| --- | --- |
| タグの有無・出現率 | 読み込んだ全曲が分母 |
| 曲ごとのタグ数 | 取得済み曲。空タグは0 |
| 重み分布 | 配布値1〜100。嗜好確率・推薦scoreではない |

- この章を主な探索に使う。fixtureの図は合成例。
- ユーザーはIDのみ。年齢・地域・嗜好などの属性分布は未分析。

In [ ]:
eda = snapshot_eda(snapshot)
display(eda["tag_coverage"])
display(eda["tags"].head(15))
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
coverage = eda["tag_coverage"]
axes[0, 0].bar(coverage["status"], coverage["n_tracks"], color=["#8b507b", "#c77936", "#256b78"])
axes[0, 0].set(title="Tag availability", ylabel="Tracks")
top_tags = eda["tags"].head(10)
axes[0, 1].barh(top_tags["tag"], top_tags["track_fraction"], color="#256b78")
axes[0, 1].invert_yaxis()
axes[0, 1].set(title="Tag occurrence among all tracks", xlabel="Track fraction", xlim=(0, 1))
axes[1, 0].hist(eda["tag_counts"], bins="auto", color="#256b78", edgecolor="white")
axes[1, 0].set(title="Tags per track (known metadata)", xlabel="Tag count", ylabel="Tracks")
axes[1, 1].hist(eda["tag_weights"], bins=10, range=(1, 101), color="#c77936", edgecolor="white")
axes[1, 1].set(title="Supplied tag weights", xlabel="Weight", ylabel="Tag assignments")
fig.suptitle(f"Catalog EDA: {snapshot.overview['dataset_key']} / {snapshot.mode}")
fig.tight_layout()
plt.show()

### 参考：全量イベントの時刻分布

- 保存済み全量集計。カタログsnapshotとは対象が異なる。
- 配布時刻のタイムゾーンは未確認。UTC変換なし。
- 1970年の1件は元LFM-2bの記載期間外。図は2005-02-14以降に拡大し、集計には残す。
- 別のJSONを指定した場合は、Data Loadingの表で規模を確認する。

In [ ]:
temporal = temporal_rows(profile)
days = [row["day"] for row in temporal]
counts = [row["n_events"] for row in temporal]
cumulative = [row["cumulative_fraction"] for row in temporal]
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(days, counts, color="#256b78", linewidth=0.8)
axes[0].set(title="Reference only: full-event temporal distribution", ylabel="Events per day")
axes[1].plot(days, cumulative, color="#256b78")
axes[1].set(ylabel="Cumulative event fraction", ylim=(0, 1.02))
plot_start = max(days[0], datetime(2005, 2, 14))
if days[-1] > plot_start:
    axes[1].set_xlim(plot_start, days[-1])
fig.autofmt_xdate()
fig.tight_layout()
plt.show()
display(profile["train"])

## 5. Data Splitting

- `MUSIC_RECSYS_SPLIT_ID` を指定した場合、保存済み分割のhash・snapshot対応を読込み時に検証する。
- 下の表・図はその分割の保存済み集計。イベント再分割・DB更新は実行しない。
- `validation` の学習側はtrain、`test` はtrain + validation。
- 分類は `cold_user → cold_item → seen → eligible`。両方coldのペアはcold_userに分類。
- `ground_truth_pairs` は既知ユーザー・既知曲の未聴正解ペア数。推薦精度ではない。


In [ ]:
if saved_split is None:
    print("保存済み分割は未選択。MUSIC_RECSYS_SPLIT_ID で指定してください。")
else:
    split_summary = saved_split["summary"]
    display(
        {
            "scope": "Selected saved split",
            "split_id": str(saved_split["id"]),
            "snapshot_id": str(saved_split["snapshot_id"]),
            "config": saved_split["config"],
        }
    )
    partitions_df = DataFrame.from_dict(split_summary["partitions"], orient="index")
    partitions_df.index.name = "partition"
    evaluation_df = DataFrame.from_dict(
        {
            phase: {
                key: value
                for key, value in stats.items()
                if key not in {"training", "exclusion_order"}
            }
            for phase, stats in split_summary["evaluation"].items()
        },
        orient="index",
    )
    evaluation_df.index.name = "evaluation_phase"
    split_validation_df = DataFrame(
        split_summary["leakage_checks"].items(), columns=["check", "result"]
    )
    display(partitions_df)
    display(evaluation_df)
    display(split_validation_df)
    display(
        DataFrame(
            [
                {"role": role, "row_count": artifact.row_count, "sha256": artifact.sha256}
                for role, artifact in saved_split["artifacts"].items()
            ]
        )
    )
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    partitions_df["events"].plot.bar(ax=axes[0], color="#256b78", rot=0)
    axes[0].set(title="Selected saved split: events", ylabel="Events")
    evaluation_df[["cold_user_pairs", "cold_item_pairs", "seen_pairs", "eligible_pairs"]].plot.bar(
        ax=axes[1],
        stacked=True,
        rot=0,
        color=["#8b507b", "#c77936", "#627d98", "#256b78"],
    )
    axes[1].set(title="Selected saved split: evaluation pairs", ylabel="Unique user-track pairs")
    fig.tight_layout()
    plt.show()

### 参考：全量EDAの分割候補

- 以下は全量イベントの候補境界・集計。上の選択済み分割とは対象が異なる。
- `validation` / `test_train_only` の学習側はtrain、`test_refit_train_validation` はtrain + validation。
- cold-start率の分母は各期間のdistinct user-track pairs。ペアがなければ率は未定義。


In [ ]:
display({"status": "Reference only: full-event candidate statistics", **profile["boundaries"]})
candidates = split_candidate_tables(profile)
display(DataFrame(candidates["events"]))
display(DataFrame(candidates["eligibility"]))
phases = candidates["eligibility"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels = [row["condition"] for row in phases]
positions = list(range(len(phases)))
for metric, offset, color, label in (
    ("cold_user_fraction", -0.18, "#8b507b", "Cold user (exclusive)"),
    ("cold_item_fraction", 0.18, "#c77936", "Cold item (exclusive)"),
):
    defined = [
        (position, row[metric]) for position, row in enumerate(phases) if row[metric] is not None
    ]
    axes[0].bar(
        [position + offset for position, _ in defined],
        [value for _, value in defined],
        width=0.36,
        color=color,
        label=label,
    )
undefined = [row["condition"] for row in phases if row["unique_pairs"] == 0]
if undefined:
    axes[0].text(
        0.02,
        0.97,
        "Undefined (no pairs): " + ", ".join(undefined),
        transform=axes[0].transAxes,
        va="top",
        fontsize=8,
    )
axes[0].set(
    title="Reference: cold-start pair fractions",
    ylabel="Fraction of unique pairs",
    xticks=positions,
    xticklabels=labels,
    ylim=(0, 1),
)
axes[0].legend()
axes[1].bar(labels, [row["novel_truth_pairs"] for row in phases], color="#256b78")
axes[1].set(title="Reference: eligible novel truth", ylabel="Unique user-track pairs")
for ax in axes:
    ax.tick_params(axis="x", rotation=15, labelsize=8)
fig.tight_layout()
plt.show()

### GroundTruthの回数・再生日数 — 保存済み参考集計

- 以下はMusic4All全量のTrain／Validation集計。選択中のfixture・snapshotの実測と混同しない。
- 関連度の数値はEvaluatorの別条件。回数・日数は観測事実で、好みの直接測定ではない。
- 正解ユーザーが少ない場合は分割・対象集合を確認し、02の最適設定へ進む前に条件を見直す。


In [ ]:
try:
    validation_conditions, validation_requests_df, validation_summary = validation_overview(
        WORKFLOW
    )
    display({"scope": "Selected experiment Validation", **validation_summary})
    display(validation_conditions)
    display(
        validation_requests_df[["history_count", "candidate_count", "positive_count"]].describe()
    )
except (ValueError, OSError) as error:
    print("探索入力を確認してください:", error)

if GROUND_TRUTH_PROFILE.exists():
    ground_truth_profile = json.loads(GROUND_TRUTH_PROFILE.read_text())
    from music_recsys.search_analysis import ground_truth_rows

    display(
        {
            "scope": "Reference: full Music4All Train/Validation; not the selected snapshot",
            "source_sha256": ground_truth_profile.get("source_parquet_sha256"),
            "train_end": ground_truth_profile.get("train_end"),
        }
    )
    ground_truth_df = ground_truth_rows(ground_truth_profile)
    display(ground_truth_df)
    if not ground_truth_df.empty:
        fig, ax = plt.subplots(figsize=(7, 3.5))
        for category, rows in ground_truth_df.groupby("category"):
            ax.plot(rows.window_days, rows.multiday_fraction, "o-", label=category)
        ax.set(
            title="Reference Validation | 2+ listening dates / positive user-track pairs",
            xlabel="Evaluation window (days)",
            ylabel="Fraction",
        )
        ax.legend()
        fig.tight_layout()
        plt.show()
else:
    print("GroundTruthの回数・日数は未集計。参考値を0で補完しません。")
print("Next: 02_analysis.ipynbで同じTOMLの条件を確認し、CLI Smoke → Validation探索。")

## 6. Summary

- 出典・版・snapshot・規模、欠損・ID・タグを確認。
- 分割指定時は保存済みの期間別件数、評価対象・正解数、リーク検証を確認。
- 全量EDAと分割候補は参考資料として分けて表示。
- 02_analysisでValidation探索・設定選択、その後03_recommendationで個別確認。
